# V-JEPA Video Anomaly Detection (VAD) on UCF-Crime

**Pipeline overview**

This notebook builds a video anomaly detector using Meta AI's **V-JEPA** (Video Joint
Embedding Predictive Architecture) as a frozen feature extractor, combined with a
simple novelty detector (for now we use **k-Nearest-Neighbours (KNN)**).

The overall workflow is:

1. **Setup** — install dependencies and clone the official V-JEPA repository.
2. **Data** — download the UCF-Crime dataset (normal + anomalous surveillance videos)
   from Kaggle.
3. **Model** — build a V-JEPA video encoder and load the official pretrained
   checkpoint.
4. **Sanity check** — run the encoder on a single video to confirm the preprocessing
   and inference pipeline work end-to-end.
5. **Feature extraction** — run the encoder over *only the normal* training videos to
   build a database of "normal behaviour" embeddings (first a simple baseline version,
   then a GPU-batched optimized version).
6. **Anomaly detection** — fit a classifier on the normal embeddings: a new
   clip is flagged as anomalous if it lies unusually far (in embedding space) from its
   nearest normal neighbours.

> **Note:** This notebook is designed to run on **Google Colab** with a GPU runtime
> (Runtime → Change runtime type → GPU). You will need a Kaggle account and a
> `KAGGLE_API_TOKEN` stored in Colab's **Secrets** (key icon in the left sidebar) for
> the dataset-download cells to work.

## 1. Environment Setup

Install the video-decoding library and import every package used throughout the
notebook (data download, video I/O, tensor ops, and the utilities from scikit-learn).

In [ ]:
# 'decord' is a fast video-reading library (frame-accurate random access),
# used later to load and sample frames from the .mp4 clips.

!pip install -q decord

In [ ]:
# --- Core imports for the whole notebook ---
# os/glob/zipfile/shutil/subprocess : filesystem + dataset download/extraction
# numpy/torch/torchvision           : tensors, image transforms
# decord (VideoReader, cpu)         : fast video frame reading
# google.colab.userdata             : reads Colab "Secrets" (e.g. Kaggle token)
# joblib                            : save/load the trained KNN detector
# sklearn (NearestNeighbors, PCA)   : anomaly-detection building blocks
# matplotlib                        : plot the anomaly-score histogram later

import os
import numpy as np
import torch
from decord import VideoReader, cpu
from torchvision import transforms
import sys
from decord import VideoReader, cpu
import os
import subprocess
from google.colab import userdata
import zipfile
import glob
import joblib
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import subprocess
from google.colab import userdata

## 2. Data Acquisition — UCF-Crime Dataset (Kaggle)

The UCF-Crime dataset is downloaded directly from Kaggle using the Kaggle CLI.
Two separate downloads are performed:

- **Training set**: only the *Normal* videos (no anomalies) — these define what
  "normal" looks like for the detector.
- **Test set**: a mix of normal + anomalous videos, whose relative paths are listed
  in an uploaded `Anomaly_Test.txt` file. The dataset's folder layout on Kaggle is
  not perfectly predictable, so the code below tries several candidate folder-prefix
  patterns per category until one works, then reuses that pattern for the rest of the
  files in the same category (to avoid repeatedly guessing).

**Prerequisite:** add your Kaggle API token as a Colab secret named
`KAGGLE_API_TOKEN`, and upload `Anomaly_Test.txt` to `/content/` before running the
second cell.

In [ ]:
# Download the 220 "Normal_Videos" training clips (part 1) from the Kaggle (for first step experiemnt, we prefer to download only first part of normal videos)
# UCF-Crime dataset one file at a time, skipping any that already exist locally.

# Kaggle authentication
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

DOWNLOAD_DIR = "/content/ucf_crime"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

DATASET = "minmints/ufc-crime-full-dataset"
FOLDER = "Training-Normal-Videos-Part-1/Training-Normal-Videos-Part-1"

downloaded = 0
failed = 0

for i in range(1, 221):
    filename = f"Normal_Videos{i:03d}_x264.mp4"
    filepath = os.path.join(DOWNLOAD_DIR, filename)

    if os.path.exists(filepath):
        print(f"✓ Already exists: {filename}")
        continue

    kaggle_path = f"{FOLDER}/{filename}"

    result = subprocess.run(
        [
            "kaggle", "datasets", "download",
            "-d", DATASET,
            "-f", kaggle_path,
            "-p", DOWNLOAD_DIR,
            "--unzip",
            "--quiet"
        ],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        # loop over expected normal-video filenames (001..220)
        downloaded += 1
        print(f"✓ Downloaded: {filename}")
    else:
        failed += 1
        print(f"✗ Not found/failed: {filename}")

print("\n==============================")
print(f"Downloaded: {downloaded}")
# skip re-downloading a file that is already on disk
print(f"Failed/missing: {failed}")
print("==============================")

# build the path of this file inside the Kaggle dataset

# Show what we actually have
videos = [
    f for f in os.listdir(DOWNLOAD_DIR)
    if f.endswith(".mp4")
]

print(f"\nTotal MP4 files in {DOWNLOAD_DIR}: {len(videos)}")

In [ ]:
# Download the TEST split (normal + anomalous categories) listed in
# Anomaly_Test.txt. Because the exact folder structure inside the Kaggle
# dataset zip is inconsistent between categories, this cell brute-forces a
# few likely folder-prefix patterns per category and remembers whichever one
# worked so it isn't re-guessed for every subsequent file in that category.

# Kaggle authentication
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

DATASET = "minmints/ufc-crime-full-dataset"
DOWNLOAD_DIR = "/content/ucf_crime_test"
LIST_FILE = "/content/Anomaly_Test.txt"   # upload this file to Colab first
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Read the list of files
with open(LIST_FILE) as f:
    rel_paths = [line.strip() for line in f if line.strip()]
print(f"{len(rel_paths)} files to download")

# Candidate folder prefixes inside the Kaggle dataset (adjust if you know the real layout).
# The dataset appears to use a doubled folder name, e.g. "Folder/Folder/file.mp4".
def candidate_prefixes(category):
    if category == "Testing_Normal_Videos_Anomaly":
        return ["Testing_Normal_Videos/Testing_Normal_Videos_Anomaly",
                "Testing_Normal_Videos_Anomaly", ""]
    prefixes = []
    for part in range(1, 5):
        name = f"Anomaly-Videos-Part-{part}"
        prefixes += [f"{name}/{name}/{category}", f"{name}/{category}"]
    prefixes += [category, f"{category}/{category}", ""]
    return prefixes

def try_download(kaggle_path, out_dir):
    result = subprocess.run(
        ["kaggle", "datasets", "download", "-d", DATASET,
         "-f", kaggle_path, "-p", out_dir, "--unzip", "--quiet"],
        capture_output=True, text=True
    )
    return result.returncode == 0

working_prefix = {}   # category -> prefix that worked last time
downloaded = skipped = 0
failed_files = []

for rel in rel_paths:
    category, filename = rel.split("/")
    out_dir = os.path.join(DOWNLOAD_DIR, category)
    os.makedirs(out_dir, exist_ok=True)
    filepath = os.path.join(out_dir, filename)

    if os.path.exists(filepath):
        skipped += 1
        print(f"✓ Already exists: {rel}")
        continue

    # Try the known-good prefix first, then the rest
    prefixes = candidate_prefixes(category)
    if category in working_prefix:
        prefixes = [working_prefix[category]] + [p for p in prefixes if p != working_prefix[category]]

    success = False
    for prefix in prefixes:
        kaggle_path = f"{prefix}/{filename}" if prefix else filename
        print(kaggle_path)
        if try_download(kaggle_path, out_dir):
            working_prefix[category] = prefix
            success = True
            break

    if success:
        downloaded += 1
        print(f"✓ Downloaded: {rel}")
    else:
        failed_files.append(rel)
        print(f"✗ Not found/failed: {rel}")

print("\n==============================")
print(f"Downloaded: {downloaded}")
print(f"Already existed: {skipped}")
print(f"Failed/missing: {len(failed_files)}")
print("==============================")

if failed_files:
    print("\nFailed files:")
    for f in failed_files:
        print("  ", f)

total = sum(len([f for f in files if f.endswith(".mp4")]) for _, _, files in os.walk(DOWNLOAD_DIR))
print(f"\nTotal MP4 files in {DOWNLOAD_DIR}: {total}")

## 3. Extract the Full Train / Test Video Archives

The Kaggle download step above produced `.zip` archives (and, for the test set,
some loose `.mp4` files nested in category subfolders). This section unpacks
everything into flat `extracted/` directories so downstream code can simply glob
for `*.mp4`.

Extracting Train Videos (Normal Videos)

In [ ]:
# Unzip every downloaded training-set archive into a single flat folder.

EXTRACT_DIR = "/content/ucf_crime/extracted"
os.makedirs(EXTRACT_DIR, exist_ok=True)

zip_files = sorted(
    glob.glob(os.path.join(DOWNLOAD_DIR, "*.zip"))
)

print(f"ZIP files found: {len(zip_files)}")

for i, zip_path in enumerate(zip_files, start=1):

    print(f"[{i}/{len(zip_files)}] Extracting {os.path.basename(zip_path)}")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)

print("\nExtraction complete.")

Extracting Test Videos

In [ ]:
# Walk every category subfolder of the test set, unzip any archives found,
# and copy any already-loose .mp4 files, all into one flat 'extracted' folder.

import os
import glob
import zipfile
import shutil

BASE_DIR = "/content/ucf_crime_test"
EXTRACT_DIR = os.path.join(BASE_DIR, "extracted")
os.makedirs(EXTRACT_DIR, exist_ok=True)

SKIP_DIRS = {"extracted"}

zip_files = []
mp4_files = []

for entry in sorted(os.listdir(BASE_DIR)):
    entry_path = os.path.join(BASE_DIR, entry)
    if os.path.isdir(entry_path) and entry not in SKIP_DIRS:
        zip_files.extend(
            glob.glob(os.path.join(entry_path, "**", "*.zip"), recursive=True)
        )
        mp4_files.extend(
            glob.glob(os.path.join(entry_path, "**", "*.mp4"), recursive=True)
        )

zip_files = sorted(zip_files)
mp4_files = sorted(mp4_files)

print(f"ZIP files found: {len(zip_files)}")
print(f"Loose MP4 files found: {len(mp4_files)}")

# --- Extract zips ---
for i, zip_path in enumerate(zip_files, start=1):
    print(f"[{i}/{len(zip_files)}] Extracting {os.path.relpath(zip_path, BASE_DIR)}")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)

# --- Copy loose mp4s ---
for i, mp4_path in enumerate(mp4_files, start=1):
    dest_path = os.path.join(EXTRACT_DIR, os.path.basename(mp4_path))
    print(f"[{i}/{len(mp4_files)}] Copying {os.path.relpath(mp4_path, BASE_DIR)}")
    shutil.copy2(mp4_path, dest_path)

print("\nExtraction and copy complete.")

In [ ]:
# List the extracted test-set video files (this `video_files` list is reused
# by the embedding-extraction code further down).

video_files = sorted(
    glob.glob(os.path.join(EXTRACT_DIR, "*.mp4"))
)

print(f"Videos found: {len(video_files)}")

print("\nFirst 10:")
for video_path in video_files[:10]:
    print(video_path)

## 4. Clone & Install V-JEPA

Clone Meta AI's official **V-JEPA** repository (which contains the model
definitions we need, e.g. `vit_large`) and install its Python dependencies.

In [ ]:
# Clone the official V-JEPA source code and move into the repo directory.

!git clone https://github.com/facebookresearch/jepa.git
%cd jepa

In [ ]:
# Install the package requirements declared by the V-JEPA repo.

%cd /content/jepa
!pip install -q -r requirements.txt

## 5. Download the Pretrained V-JEPA Checkpoint (for now we use ViT-L/16)

Fetch Meta's official pretrained V-JEPA weights (for now `vit_large`, patch size 16) and
inspect the raw checkpoint dictionary to see what it contains (e.g. an `"encoder"`
key with the backbone's state dict).

In [ ]:
# Download the official pretrained V-JEPA ViT-L/16 checkpoint.

!wget -O /content/vitl16.pth.tar https://dl.fbaipublicfiles.com/jepa/vitl16/vitl16.pth.tar

In [ ]:
# Load the checkpoint onto CPU first and inspect its top-level structure
# (it's a dict containing, among other things, an 'encoder' state dict).

checkpoint = torch.load(
    "/content/vitl16.pth.tar",
    map_location="cpu"
)

print(type(checkpoint))
print(checkpoint.keys())

## 6. Build the Model Architecture & Load Encoder Weights

Instantiate the `vit_large` V-JEPA video transformer with the same architecture
hyperparameters used at pretraining time (224×224 input, 16×16 patches, 16-frame
clips, tubelet size 2), then load the pretrained encoder weights into it. The
checkpoint's key names are prefixed with `module.backbone.` (an artifact of how it
was originally saved, e.g. under `DistributedDataParallel`), so that prefix is
stripped before loading.

In [ ]:
# Make the cloned repo importable, then build the V-JEPA ViT-Large backbone.
# These hyperparameters must match how the checkpoint was pretrained.

sys.path.append("/content/jepa")

from src.models.vision_transformer import vit_large

model = vit_large(
    img_size=224,
    patch_size=16,
    num_frames=16,
    tubelet_size=2,
)

print(model)

In [ ]:
# Extract just the encoder weights from the checkpoint and strip the
# 'module.backbone.' prefix so the keys line up with our plain `model`.

# load the enocder weights

encoder_state = checkpoint["encoder"]

encoder_state = {
    key.replace("module.backbone.", "", 1): value
    for key, value in encoder_state.items()
}

missing, unexpected = model.load_state_dict(
    encoder_state,
    strict=True
)

print("Weights loaded successfully!")
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

## 7. Walkthrough: Extract an Embedding for a Single Video

Before processing the whole dataset, this section runs the full pipeline
(read video → sample frames → preprocess → forward pass through the encoder) on
**one** example video, purely as a sanity check that every step produces the
expected tensor shapes.

In [ ]:
# Pick one example training video to test the pipeline on.
video_path = "/content/ucf_crime/extracted/Normal_Videos001_x264.mp4"

In [ ]:
# Open the video and print basic metadata (frame count, FPS, frame resolution).
vr = VideoReader(video_path, ctx=cpu(0))

print("Frames:", len(vr))
print("FPS:", vr.get_avg_fps())
print("Resolution:", vr[0].shape)

In [ ]:
# Uniformly sample 16 frames spaced 4 frames apart (i.e. a 61-frame span of the
# original video is compressed down to a 16-frame clip fed to the model).
num_frames = 16
sampling_rate = 4

total_needed = (num_frames - 1) * sampling_rate + 1

start = 0

frame_indices = start + np.arange(num_frames) * sampling_rate

frames = vr.get_batch(frame_indices).asnumpy()

print("Frames shape:", frames.shape)

In [ ]:
# Convert the sampled frames into a normalized model-ready tensor:
#   1) uint8 (T,H,W,C) -> float (T,C,H,W) in [0, 1]
#   2) resize + center-crop to 224x224, then ImageNet mean/std normalize
#   3) rearrange to (C,T,H,W) and add a batch dimension -> (1,C,T,H,W)


# Convert (T, H, W, C) → (T, C, H, W)
frames_tensor = torch.from_numpy(frames).permute(0, 3, 1, 2).float() / 255.0

# Resize the short side to 224, then center crop to 224×224
preprocess = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

frames_tensor = preprocess(frames_tensor)

# (T, C, H, W) → (C, T, H, W) → add batch dimension
video_tensor = frames_tensor.permute(1, 0, 2, 3).unsqueeze(0)
print("Input shape:", video_tensor.shape)

In [ ]:
# Use the GPU if Colab has one attached, otherwise fall back to CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# Move the model's parameters onto the selected device.
model = model.to(device)
print("Model device:", next(model.parameters()).device)

In [ ]:
# Move the input clip to the same device and run a forward pass to get
# the V-JEPA embedding for this single clip (no gradient tracking needed).
video_tensor = video_tensor.to(device)

with torch.inference_mode():
    features = model(video_tensor)

print("Feature embedding:", features.shape())

## 9. Prepare the Model for Batch Inference

Put the encoder in evaluation mode, freeze all parameters (no training happens
here — V-JEPA is used purely as a frozen feature extractor), and re-declare the
frame-sampling / preprocessing constants used by the embedding-extraction
functions below.

In [ ]:
# eval() disables dropout/batchnorm-training behaviour; requires_grad=False
# freezes every weight since we only need forward passes, not training.

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

for parameter in model.parameters():
    parameter.requires_grad = False

print("Device:", device)
print("Model ready for inference.")

In [ ]:
# Same clip-sampling settings as the single-video test above:
# 16 frames per clip, spaced 4 frames apart -> each clip spans 61 original frames.

NUM_FRAMES = 16
SAMPLING_RATE = 4

preprocess = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

TOTAL_FRAMES_PER_CLIP = (
    (NUM_FRAMES - 1) * SAMPLING_RATE + 1
)

print("Frames sampled per clip:", NUM_FRAMES)
print("Frame sampling rate:", SAMPLING_RATE)
print("Original-video span per clip:", TOTAL_FRAMES_PER_CLIP)

## 10. Baseline Embedding-Extraction Function

`extract_video_embedding` turns an entire video into a single fixed-length
embedding:

1. Split the video into consecutive, non-overlapping 61-frame spans.
2. For each span, sample 16 frames, preprocess, and run the encoder to get a
   `(1568, 1024)` token-embedding grid.
3. Mean-pool over the 1568 spatial/temporal tokens → one 1024-d vector per clip.
4. Mean-pool across all clips in the video → one 1024-d vector per **video**.

This version processes one clip at a time on the GPU/CPU and is simple but slow —
it's kept here as the readable reference implementation before the optimized
version below.

In [ ]:
# Reference (unoptimized) implementation: processes clips one at a time.

def extract_video_embedding(video_path):
    vr = VideoReader(video_path, ctx=cpu(0))

    total_video_frames = len(vr)

    video_embeddings = []

    for start in range(
        0,
        total_video_frames - TOTAL_FRAMES_PER_CLIP + 1,
        TOTAL_FRAMES_PER_CLIP
    ):

        frame_indices = (
            start
            + np.arange(NUM_FRAMES) * SAMPLING_RATE
        )

        frames = vr.get_batch(frame_indices).asnumpy()

        # (T, H, W, C) -> (T, C, H, W)
        frames_tensor = (
            torch.from_numpy(frames)
            .permute(0, 3, 1, 2)
            .float()
            / 255.0
        )

        frames_tensor = preprocess(frames_tensor)

        # (T,C,H,W) -> (C,T,H,W) -> (1,C,T,H,W)
        video_tensor = (
            frames_tensor
            .permute(1, 0, 2, 3)
            .unsqueeze(0)
            .to(device)
        )

        with torch.inference_mode():
            features = model(video_tensor)

        # features: (1, 1568, 1024)
        features = features.squeeze(0)          # (1568, 1024)

        # Mean-pool the 1568 tokens
        clip_embedding = features.mean(dim=0)   # (1024,)

        # clip_embedding = [1024]
        video_embeddings.append(
            clip_embedding.cpu()
        )

    if len(video_embeddings) == 0:
        return None

    # [number_of_clips, 1024]
    video_embeddings = torch.stack(video_embeddings)

    # Mean-pool the clips
    video_embedding = video_embeddings.mean(dim=0)

    # [1024]
    return video_embedding.numpy()

# Optimzed Version - (AI GENERATED)

This optimized version speeds things up (while returning the *same* final
per-video embedding) by:

- **Batching multiple clips** per forward pass (`CLIPS_PER_BATCH`, default 8)
  instead of one clip at a time.
- **Preprocessing on the GPU** (`_preprocess_gpu`) — the resize/crop/normalize
  steps run as vectorized GPU tensor ops instead of per-frame CPU `torchvision`
  transforms.
- Reading all frames for a batch of clips in a **single decord `get_batch` call**
  and keeping data as `uint8` until it's on the GPU, reducing CPU/decoding overhead.
- Running inference under **mixed precision (`float16` autocast)** on CUDA for a
  further speed/memory improvement.
- Optionally returning the **per-clip embeddings** as well (`return_clips=True`),
  which isn't used later in this notebook but is handy for debugging/analysis.

In [ ]:
# GPU-batched, mixed-precision replacement for extract_video_embedding.
# Produces the same final (1024,) per-video embedding as the baseline above,
# just far faster.

import torch.nn.functional as F

CLIPS_PER_BATCH = 8   # lower to 4 if CUDA out-of-memory

_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


def _preprocess_gpu(frames):
    # uint8 (B, T, H, W, 3) on GPU -> float (B, 3, T, 224, 224)
    B, T, H, W, _ = frames.shape
    x = frames.view(B * T, H, W, 3).permute(0, 3, 1, 2).float().div_(255)
    s = 224 / min(H, W)
    nh, nw = round(H * s), round(W * s)
    x = F.interpolate(x, size=(nh, nw), mode="bilinear", antialias=True, align_corners=False)
    top, left = (nh - 224) // 2, (nw - 224) // 2
    x = x[:, :, top:top + 224, left:left + 224]
    x = (x - _mean) / _std
    return x.view(B, T, 3, 224, 224).permute(0, 2, 1, 3, 4).contiguous()


def extract_video_embedding(video_path, return_clips=False):
    vr = VideoReader(video_path, ctx=cpu(0))

    starts = list(range(0, len(vr) - TOTAL_FRAMES_PER_CLIP + 1, TOTAL_FRAMES_PER_CLIP))
    if len(starts) == 0:
        return None

    clip_embs = []
    for i in range(0, len(starts), CLIPS_PER_BATCH):
        s = np.array(starts[i:i + CLIPS_PER_BATCH])
        idx = (s[:, None] + np.arange(NUM_FRAMES)[None, :] * SAMPLING_RATE).ravel()

        # one decode call for the whole batch, keep uint8 until it is on the GPU
        frames = torch.from_numpy(vr.get_batch(idx).asnumpy())
        frames = frames.view(len(s), NUM_FRAMES, *frames.shape[1:]).to(device)

        x = _preprocess_gpu(frames)

        with torch.inference_mode(), torch.autocast(
            device.type, dtype=torch.float16, enabled=(device.type == "cuda")
        ):
            feats = model(x)                              # (B, 1568, 1024)

        clip_embs.append(feats.float().mean(dim=1).cpu())  # (B, 1024)

    clip_embs = torch.cat(clip_embs)                      # (n_clips, 1024)
    video_emb = clip_embs.mean(dim=0).numpy()             # (1024,) same as before

    return (video_emb, clip_embs.numpy()) if return_clips else video_emb

## 12. Quick Test of the Optimized Extractor

Run the new optimized `extract_video_embedding` on a single test video to confirm
it executes correctly before using it on the full dataset.

In [ ]:
# Smoke-test the optimized extractor on one video before running it on all of them.

test_video = video_files[52]

print("Testing:", test_video)

test_embeddings = extract_video_embedding(
    test_video
)

print(
    "Embedding shape:",
    test_embeddings
)

## 13. Build the "Normal Behaviour" Embedding Database

Run the extractor over **every** normal training video, collect one 1024-d
embedding per video (label `0` = normal), and save the resulting embedding matrix
+ labels to disk (`normal_embeddings.npz`). This embedding set is what the KNN
anomaly detector will later be fit on.

In [ ]:
# Extract an embedding for every normal training video and persist the
# resulting (N, 1024) embedding matrix + labels to disk.

embeddings = []
labels = []

for video in video_files:
    embedding = extract_video_embedding(video)

    print(f"{video} : {embedding}")

    embeddings.append(embedding)
    labels.append(0)  # 0 = normal

embeddings = np.array(embeddings)
labels = np.array(labels)

np.savez(
    "normal_embeddings.npz",
    embeddings=embeddings,
    labels=labels
)

print("Embeddings shape:", embeddings.shape)
print("Labels shape:", labels.shape)

## 14. Reload the Saved Embeddings

Load the embeddings back from disk into `X_normal` (features) and `y` (labels),
so the anomaly-detector training below can be re-run independently of the
(expensive) extraction step above.

In [ ]:
# Reload the previously saved normal-video embeddings and labels.

data = np.load("normal_embeddings.npz")
X_normal=data["embeddings"]
y=data["labels"]
y.shape

## 15. KNN-Based Anomaly Detector

`KNNAnomalyDetector` is a lightweight novelty-detection model:

- **Fit**: given only *normal* embeddings, optionally L2-normalize (and/or
  PCA-reduce) them, then compute each training point's average distance to its
  `k` nearest neighbours **excluding itself** (leave-one-out). The
  `threshold_percentile`-th percentile of these distances becomes the anomaly
  threshold.
- **Score**: for a new embedding, the anomaly score is its average distance to
  the `k` nearest *normal* training embeddings — higher means more unusual.
- **Predict**: an embedding is flagged as an anomaly (`1`) if its score exceeds
  the fitted threshold, otherwise normal (`0`).

In [ ]:
# A minimal k-NN novelty detector: trained on normal data only, flags
# points whose neighbourhood distance is unusually large as anomalies.

class KNNAnomalyDetector:
    def __init__(self, k=10, metric="cosine", normalize=True,
                 pca_components=None, threshold_percentile=95.0):
        """
        k                    : neighbours used for the score (3-10 is good for small data)
        metric               : 'cosine' or 'euclidean'
        normalize            : L2-normalise embeddings first (recommended)
        pca_components       : int / float in (0,1) to reduce dims, or None
        threshold_percentile : percentile of normal (leave-one-out) scores used as threshold
        """
        self.k = k
        self.metric = metric
        self.normalize = normalize
        self.pca_components = pca_components
        self.threshold_percentile = threshold_percentile

    def _prep(self, X, fit=False):
        X = np.asarray(X, dtype=np.float32)
        if self.normalize:
            X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
        if self.pca_components is not None:
            if fit:
                self.pca_ = PCA(n_components=self.pca_components, random_state=0)
                X = self.pca_.fit_transform(X)
            else:
                X = self.pca_.transform(X)
        return X

    def fit(self, X_normal):
        Xp = self._prep(X_normal, fit=True)
        self.nn_ = NearestNeighbors(metric=self.metric, algorithm="brute").fit(Xp)
        # Leave-one-out: drop each point's match with itself (distance 0),
        # otherwise the threshold would be far too tight.
        dist, _ = self.nn_.kneighbors(Xp, n_neighbors=self.k + 1)
        self.train_scores_ = dist[:, 1:].mean(axis=1)
        self.threshold_ = float(np.percentile(self.train_scores_, self.threshold_percentile))
        return self

    def score(self, X):
        """Anomaly score: higher = more anomalous."""
        dist, _ = self.nn_.kneighbors(self._prep(X), n_neighbors=self.k)
        return dist.mean(axis=1)

    def predict(self, X):
        """1 = anomaly, 0 = normal."""
        return (self.score(X) > self.threshold_).astype(int)

## 16. Hyperparameter Sanity Check (sweep over `k`)

Split the normal embeddings into an 80/20 train/held-out split, then fit the
detector with a few candidate values of `k` and report the **false-alarm rate**
on the held-out *normal* videos (i.e. how often normal data is wrongly flagged
as anomalous) at a strict 99th-percentile threshold. This is a quick, informal
way to pick a reasonable `k` before training the final model on all the data.

In [ ]:
# Randomly split normal embeddings 80/20 and compare a few values of k by
# their false-alarm rate on the held-out normal videos (lower is better).

rng = np.random.default_rng(0)
idx = rng.permutation(len(X_normal))
n_tr = int(0.8 * len(idx))
tr, te = X_normal[idx[:n_tr]], X_normal[idx[n_tr:]]

for k in (3, 5, 10):
    d = KNNAnomalyDetector(k=k, threshold_percentile=99).fit(tr)
    print(f"k={k:2d}  threshold={d.threshold_:.3f}  held-out normal false-alarm rate={d.predict(te).mean():.2%}")

## 17. Train the Final Detector & Visualize the Score Distribution

Fit the detector on **all** normal embeddings with the chosen `k` and
`threshold_percentile`, plot a histogram of the leave-one-out normal anomaly
scores together with the chosen decision threshold, and save the trained
detector to disk (`knn_detector.joblib`) for later reuse (e.g. scoring the test
set / new videos).

In [ ]:
# Train the final KNN anomaly detector on the full normal-embedding set,
# visualize where the threshold falls relative to normal scores, and save it.

K = 5
PERCENTILE = 97      # higher (e.g. 99) = fewer false alarms, lower = more sensitive

det = KNNAnomalyDetector(k=K, metric="cosine", threshold_percentile=PERCENTILE).fit(X_normal)
print(f"Trained on {len(X_normal)} normal embeddings | threshold = {det.threshold_:.4f}")

plt.figure(figsize=(7, 3.5))
plt.hist(det.train_scores_, bins=30, alpha=.8, label="normal (leave-one-out)")
plt.axvline(det.threshold_, color="r", ls="--", label=f"threshold ({PERCENTILE}th pct)")
plt.xlabel("anomaly score"); plt.ylabel("count"); plt.legend(); plt.tight_layout(); plt.show()

joblib.dump(det, "knn_detector.joblib")   # download via the Files panel or files.download(...)